# FIFA World Cup 2026 — Stadiums Data Collection

This notebook collects the official FIFA World Cup 2026 stadium data from the FIFA API.

Source:
FIFA World Cup 2026 official stadium article API.

The collected data will later be cleaned, validated, and saved as `Stadiums.csv`.

In [1]:
import requests
import pandas as pd
import re

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. FIFA API Connection

We will connect to the official FIFA World Cup 2026 article API.

The API returns structured information about the 16 official World Cup 2026 stadiums.

In [2]:
stadium_api_url = (
    "https://cxm-api.fifa.com/fifaplusweb/api/"
    "sections/article/6ICHDVRRefwZVJT2w6KCFU?locale=en"
)

response = requests.get(
    stadium_api_url,
    headers={
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/139.0.0.0 Safari/537.36"
        )
    },
    timeout=30
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response size:", len(response.content), "bytes")

Status code: 200
Content type: application/json; charset=utf-8
Response size: 109699 bytes


## 3. Read Stadium Data

The FIFA API response is returned as JSON.

We will convert the response into a Python dictionary so we can inspect its structure before extracting the stadium information.

In [3]:
stadium_data = response.json()

print("JSON response loaded successfully.")
print("Data type:", type(stadium_data))
print("Top-level keys:")
print(stadium_data.keys())

JSON response loaded successfully.
Data type: <class 'dict'>
Top-level keys:
dict_keys(['entryType', 'entryId', 'articleTitle', 'articlePublishedDate', 'heroImage', 'richtext', 'publishedLabel', 'backLabel', 'shareLabel', 'authorLabel', 'theme', 'customTheme', 'articlePreviewText', 'previewImage', 'articlePageInternalTitle'])


## 4. Inspect the Stadium Richtext Structure

The stadium details are stored inside the FIFA API's `richtext` field.

We will inspect the structure of this field to identify how stadium names, locations, and capacities are represented.

In [4]:
richtext_data = stadium_data["richtext"]

print("Richtext data type:", type(richtext_data))

if isinstance(richtext_data, list):
    print("Number of richtext items:", len(richtext_data))
    print("\nFirst item:")
    print(richtext_data[0])
else:
    print("Richtext structure:")
    print(richtext_data)

Richtext data type: <class 'dict'>
Richtext structure:
{'nodeType': 'document', 'data': {}, 'content': [{'nodeType': 'unordered-list', 'data': {}, 'content': [{'nodeType': 'list-item', 'data': {}, 'content': [{'nodeType': 'paragraph', 'data': {}, 'content': [{'nodeType': 'text', 'data': {}, 'value': 'Canada, Mexico and the USA to host tournament\ufeff', 'marks': [{'type': 'bold'}]}]}]}, {'nodeType': 'list-item', 'data': {}, 'content': [{'nodeType': 'paragraph', 'data': {}, 'content': [{'nodeType': 'text', 'data': {}, 'value': '16 stadiums will host FIFA World Cup 2026, the most since 2002\ufeff', 'marks': [{'type': 'bold'}]}]}]}, {'nodeType': 'list-item', 'data': {}, 'content': [{'nodeType': 'paragraph', 'data': {}, 'content': [{'nodeType': 'text', 'data': {}, 'value': 'FIFA details all you need to know about the venues', 'marks': [{'type': 'bold'}]}]}]}]}, {'nodeType': 'paragraph', 'data': {}, 'content': [{'nodeType': 'text', 'data': {}, 'value': 'The ', 'marks': []}, {'nodeType': 'hy

## 5. Access the Article Content

The stadium information is stored inside the `content` list of the `richtext` dictionary.

We will access this list and inspect the number and types of content blocks returned by FIFA.

In [5]:
article_content = stadium_data["richtext"]["content"]

print("Article content loaded successfully.")
print("Number of content blocks:", len(article_content))

print("\nFirst 10 node types:")
for node in article_content[:10]:
    print(node["nodeType"])

Article content loaded successfully.
Number of content blocks: 155

First 10 node types:
unordered-list
paragraph
paragraph
paragraph
embedded-entry-block
hr
paragraph
hr
heading-2
heading-3


## 6. Identify Stadium Names

FIFA stores each stadium name as a `heading-3` node.

We will extract the text from all `heading-3` nodes to identify the stadium entries in the article.

In [6]:
stadium_headings = []

for node in article_content:
    if node["nodeType"] == "heading-3":
        text = ""

        for item in node.get("content", []):
            if item["nodeType"] == "text":
                text += item.get("value", "")
            elif item["nodeType"] == "hyperlink":
                for link_item in item.get("content", []):
                    if link_item["nodeType"] == "text":
                        text += link_item.get("value", "")

        text = text.strip().replace("\ufeff", "")

        if text:
            stadium_headings.append(text)

print("Number of heading-3 entries:", len(stadium_headings))

print("\nHeading-3 entries:")
for i, stadium in enumerate(stadium_headings, start=1):
    print(f"{i}. {stadium}")

Number of heading-3 entries: 17

Heading-3 entries:
1. Toronto Stadium
2. BC Place Vancouver
3. Mexico City Stadium
4. Guadalajara Stadium
5. Estadio Monterrey
6. USA
7. Atlanta Stadium
8. Boston Stadium
9. Dallas Stadium
10. Houston Stadium
11. Kansas City Stadium
12. Los Angeles Stadium
13. Miami Stadium
14. New York New Jersey Stadium
15. Philadelphia Stadium
16. San Francisco Bay Area Stadium
17. Seattle Stadium


## 7. Filter Stadium Names

The FIFA article contains country headings using the same `heading-3` node type.

The entry `USA` is a country heading, not a stadium, so we will remove it and keep the 16 actual stadiums.

In [7]:
country_headings = {"USA", "Canada", "Mexico"}

stadium_names = [
    stadium
    for stadium in stadium_headings
    if stadium not in country_headings
]

print("Number of stadiums:", len(stadium_names))

print("\nStadium names:")
for i, stadium in enumerate(stadium_names, start=1):
    print(f"{i}. {stadium}")

Number of stadiums: 16

Stadium names:
1. Toronto Stadium
2. BC Place Vancouver
3. Mexico City Stadium
4. Guadalajara Stadium
5. Estadio Monterrey
6. Atlanta Stadium
7. Boston Stadium
8. Dallas Stadium
9. Houston Stadium
10. Kansas City Stadium
11. Los Angeles Stadium
12. Miami Stadium
13. New York New Jersey Stadium
14. Philadelphia Stadium
15. San Francisco Bay Area Stadium
16. Seattle Stadium


## 8. Extract Stadium Details

For each stadium heading, FIFA provides a following paragraph containing:

- Location
- Capacity
- Opening year

We will extract the Location and Capacity values needed for our `Stadiums` SQL table.

In [8]:
stadium_details = []

for i, node in enumerate(article_content):
    if node["nodeType"] == "heading-3":
        
        # Extract heading text
        stadium_name = ""

        for item in node.get("content", []):
            if item["nodeType"] == "text":
                stadium_name += item.get("value", "")
            
            elif item["nodeType"] == "hyperlink":
                for link_item in item.get("content", []):
                    if link_item["nodeType"] == "text":
                        stadium_name += link_item.get("value", "")

        stadium_name = stadium_name.strip().replace("\ufeff", "")

        # Skip country headings
        if stadium_name in {"USA", "Canada", "Mexico"}:
            continue

        # Find the next paragraph
        for next_node in article_content[i + 1:]:
            if next_node["nodeType"] == "paragraph":
                
                paragraph_text = ""

                for item in next_node.get("content", []):
                    if item["nodeType"] == "text":
                        paragraph_text += item.get("value", "")
                
                paragraph_text = paragraph_text.replace("\ufeff", "")

                # Make sure this is the stadium details paragraph
                if "Location:" in paragraph_text and "Capacity:" in paragraph_text:
                    stadium_details.append({
                        "StadiumName": stadium_name,
                        "Details": paragraph_text
                    })
                    break

print("Stadium detail records found:", len(stadium_details))

for stadium in stadium_details:
    print("\n", stadium["StadiumName"])
    print(stadium["Details"])

Stadium detail records found: 16

 Toronto Stadium
Location: Toronto, Ontario, Canada
Capacity: 43,036
Opened: 2007

 BC Place Vancouver
Location: Vancouver, British Colombia, Canada
Capacity: 52,497
Opened: 1983

 Mexico City Stadium
Location: Mexico City, Mexico
Capacity: 80,824
Opened: 1966

 Guadalajara Stadium
Location: Zapopan, Jalisco, Mexico
Capacity: 45,664
Opened: 2010

 Estadio Monterrey
Location: Guadalupe, Nuevo Leon, Mexico
Capacity: 51,243
Opened: 2015

 Atlanta Stadium
Location: Atlanta, Georgia, USA
Capacity: 68,239
Opened: 2017

 Boston Stadium
Location: Foxborough, Massachusetts, USA
Capacity: 64,146
Opened: 2002

 Dallas Stadium
Location: Arlington, Texas, USA
Capacity: 70,649
Opened: 2009

 Houston Stadium
Location: Houston, Texas, USA
Capacity: 68,777
Opened: 2002

 Kansas City Stadium
Location: Kansas City, Missouri, USA
Capacity: 69,045
Opened: 1972

 Los Angeles Stadium
Location: Inglewood, California, USA
Capacity: 70,492
Opened: 2020

 Miami Stadium
Location:

## 9. Parse Location and Capacity

The raw FIFA stadium details contain Location and Capacity in text form.

We will extract these values into separate columns so the dataset matches the SQL `Stadiums` table structure.

In [9]:
parsed_stadiums = []

for stadium in stadium_details:
    details = stadium["Details"]

    location_match = re.search(
        r"Location:\s*(.*?)(?:\n|Capacity:)",
        details
    )

    capacity_match = re.search(
        r"Capacity:\s*([\d,]+)",
        details
    )

    parsed_stadiums.append({
        "StadiumName": stadium["StadiumName"],
        "Location": location_match.group(1).strip()
        if location_match else None,
        "Capacity": int(capacity_match.group(1).replace(",", ""))
        if capacity_match else None
    })

stadiums_df = pd.DataFrame(parsed_stadiums)

print("Parsed stadium records:", len(stadiums_df))
print("\nColumns:")
print(stadiums_df.columns.tolist())

print("\nFirst 5 records:")
print(stadiums_df.head())

Parsed stadium records: 16

Columns:
['StadiumName', 'Location', 'Capacity']

First 5 records:
           StadiumName                             Location  Capacity
0      Toronto Stadium             Toronto, Ontario, Canada     43036
1   BC Place Vancouver  Vancouver, British Colombia, Canada     52497
2  Mexico City Stadium                  Mexico City, Mexico     80824
3  Guadalajara Stadium             Zapopan, Jalisco, Mexico     45664
4    Estadio Monterrey        Guadalupe, Nuevo Leon, Mexico     51243


## 10. Split Location into City and Country

The FIFA API provides the location as a combined value.

We will separate it into City and Country to match the structure of the SQL `Stadiums` table.

In [10]:
country_names = ["Canada", "Mexico", "USA"]

def split_location(location):
    for country in country_names:
        if location.endswith(country):
            city = location[:-(len(country))].rstrip(", ")
            return city, country
    
    return None, None


stadiums_df[["City", "Country"]] = (
    stadiums_df["Location"]
    .apply(lambda x: pd.Series(split_location(x)))
)

stadiums_df = stadiums_df[
    ["StadiumName", "City", "Country", "Capacity"]
]

print("Columns:")
print(stadiums_df.columns.tolist())

print("\nFirst 5 records:")
print(stadiums_df.head())

Columns:
['StadiumName', 'City', 'Country', 'Capacity']

First 5 records:
           StadiumName                         City Country  Capacity
0      Toronto Stadium             Toronto, Ontario  Canada     43036
1   BC Place Vancouver  Vancouver, British Colombia  Canada     52497
2  Mexico City Stadium                  Mexico City  Mexico     80824
3  Guadalajara Stadium             Zapopan, Jalisco  Mexico     45664
4    Estadio Monterrey        Guadalupe, Nuevo Leon  Mexico     51243


## 11. Create StadiumID

The SQL `Stadiums` table requires a unique `StadiumID`.

We will create our own sequential IDs from 1 to 16, independent of any FIFA internal IDs.

In [11]:
stadiums_df.insert(
    0,
    "StadiumID",
    range(1, len(stadiums_df) + 1)
)

print("StadiumID created successfully.")
print("\nFirst 5 records:")
print(stadiums_df.head())

print("\nStadiumID range:")
print(stadiums_df["StadiumID"].min(), "to", stadiums_df["StadiumID"].max())

StadiumID created successfully.

First 5 records:
   StadiumID          StadiumName                         City Country  \
0          1      Toronto Stadium             Toronto, Ontario  Canada   
1          2   BC Place Vancouver  Vancouver, British Colombia  Canada   
2          3  Mexico City Stadium                  Mexico City  Mexico   
3          4  Guadalajara Stadium             Zapopan, Jalisco  Mexico   
4          5    Estadio Monterrey        Guadalupe, Nuevo Leon  Mexico   

   Capacity  
0     43036  
1     52497  
2     80824  
3     45664  
4     51243  

StadiumID range:
1 to 16


## 12. Stadium Dataset Validation

We will validate the stadium dataset before exporting it.

Checks:
- Exactly 16 stadiums
- StadiumID values are unique
- Stadium names are unique
- No missing values
- Capacity values are positive
- All three host countries are represented

In [12]:
print("=== Stadium Dataset Validation ===")

checks = {
    "Exactly 16 stadiums": len(stadiums_df) == 16,
    "Unique StadiumIDs": stadiums_df["StadiumID"].is_unique,
    "Unique StadiumNames": stadiums_df["StadiumName"].is_unique,
    "No missing values": not stadiums_df.isnull().any().any(),
    "All capacities positive": (stadiums_df["Capacity"] > 0).all(),
    "Canada represented": (stadiums_df["Country"] == "Canada").any(),
    "Mexico represented": (stadiums_df["Country"] == "Mexico").any(),
    "USA represented": (stadiums_df["Country"] == "USA").any()
}

for check, result in checks.items():
    print(f"{check}: {result}")

print("\nTotal rows:", len(stadiums_df))
print("Total columns:", len(stadiums_df.columns))

print("\nCountry distribution:")
print(stadiums_df["Country"].value_counts())

print("\nOverall validation:",
      "PASSED" if all(checks.values()) else "FAILED")

=== Stadium Dataset Validation ===
Exactly 16 stadiums: True
Unique StadiumIDs: True
Unique StadiumNames: True
No missing values: True
All capacities positive: True
Canada represented: True
Mexico represented: True
USA represented: True

Total rows: 16
Total columns: 5

Country distribution:
Country
USA       11
Mexico     3
Canada     2
Name: count, dtype: int64

Overall validation: PASSED


## 13. Final Stadium Dataset Preview

We will display the complete stadium dataset to perform a final visual check before exporting it to CSV.

In [13]:
pd.set_option("display.max_rows", None)

display(stadiums_df)

,StadiumID,StadiumName,City,Country,Capacity
0,1,Toronto Stadium,"Toronto, Ontario",Canada,43036
1,2,BC Place Vancouver,"Vancouver, British Colombia",Canada,52497
2,3,Mexico City Stadium,Mexico City,Mexico,80824
3,4,Guadalajara Stadium,"Zapopan, Jalisco",Mexico,45664
4,5,Estadio Monterrey,"Guadalupe, Nuevo Leon",Mexico,51243
5,6,Atlanta Stadium,"Atlanta, Georgia",USA,68239
6,7,Boston Stadium,"Foxborough, Massachusetts",USA,64146
7,8,Dallas Stadium,"Arlington, Texas",USA,70649
8,9,Houston Stadium,"Houston, Texas",USA,68777
9,10,Kansas City Stadium,"Kansas City, Missouri",USA,69045


## 14. Export Stadiums Dataset

The validated stadium dataset will be exported as `Stadiums.csv` inside the project's `Data` folder.

The path is built dynamically so the notebook remains portable across computers.

In [14]:
from pathlib import Path

project_folder = Path.cwd().parent
data_folder = project_folder / "Data"
data_folder.mkdir(parents=True, exist_ok=True)

stadiums_csv_path = data_folder / "Stadiums.csv"

stadiums_df.to_csv(
    stadiums_csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("Stadiums dataset exported successfully.")
print("File location:")
print(stadiums_csv_path)

Stadiums dataset exported successfully.
File location:
d:\ai course hw\FIFA_WorldCup_2026_Project\Data\Stadiums.csv


## 15. Verify Exported Stadiums CSV

We will read the exported CSV file back into Python and verify its structure and row count.

In [15]:
stadiums_check_df = pd.read_csv(stadiums_csv_path)

print("CSV file exists:", stadiums_csv_path.exists())
print("CSV shape:", stadiums_check_df.shape)
print("CSV columns:", stadiums_check_df.columns.tolist())

print("\nFirst 5 rows:")
display(stadiums_check_df.head())

print("\nNumber of stadiums:", len(stadiums_check_df))

CSV file exists: True
CSV shape: (16, 5)
CSV columns: ['StadiumID', 'StadiumName', 'City', 'Country', 'Capacity']

First 5 rows:


,StadiumID,StadiumName,City,Country,Capacity
0,1,Toronto Stadium,"Toronto, Ontario",Canada,43036
1,2,BC Place Vancouver,"Vancouver, British Colombia",Canada,52497
2,3,Mexico City Stadium,Mexico City,Mexico,80824
3,4,Guadalajara Stadium,"Zapopan, Jalisco",Mexico,45664
4,5,Estadio Monterrey,"Guadalupe, Nuevo Leon",Mexico,51243



Number of stadiums: 16
